# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-morad15/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal checks

**Signal 1 — GSC impressions: OPPOSITE**

Higher February impressions did not correspond to higher March review priority in this slice. Review priority was highest in the 100–499 bucket (52.75%) and decreased to 27.03% in the 5k+ bucket, so impressions are not used as a positive scoring signal.

**Signal 2 — GSC average position: CONFIRMED**

Higher February average position was strongly associated with higher March review priority in this slice. Review priority increased from 15.11% for positions 1–3 to 92.40% for positions 50+, so position is used as the main signal in the baseline.

### Baseline rule

Prioritize pages for review when their February Google Search Console average position is poor. The score increases with worse average position, while pages with better positions receive lower priority.

The rule uses February decision-time information only. March data is used only to evaluate the baseline, not to calculate its score.

### Reason codes

- `position_slipping` — poor February average position.
- `lower_priority` — better February average position.

In [1]:
import pandas as pd
import numpy as np
import duckdb

con = duckdb.connect()

print("DuckDB ready.")

DuckDB ready.


In [2]:
FEB_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
"""

MAR_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [4]:
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("DuckDB Hugging Face authentication configured.")

DuckDB Hugging Face authentication configured.


In [5]:
baseline_df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_sum_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position
    FROM {FEB_REL}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_sum_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS march_avg_position
    FROM {MAR_REL}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    feb.client_hash_id,
    feb.content_hash_id,
    feb.gsc_impressions,
    feb.gsc_clicks,
    feb.gsc_avg_position,
    mar.march_avg_position,

    CASE
        WHEN mar.march_avg_position > 10 THEN 1
        ELSE 0
    END AS review_priority

FROM feb
INNER JOIN mar
    USING (client_hash_id, content_hash_id)

WHERE feb.gsc_avg_position IS NOT NULL
  AND mar.march_avg_position IS NOT NULL
""").df()

print("Baseline audit shape:", baseline_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline audit shape: (134238, 7)


In [6]:
signal_1 = (
    baseline_df.assign(
        impressions_bucket=pd.cut(
            baseline_df["gsc_impressions"],
            bins=[-1, 99, 499, 999, 4999, float("inf")],
            labels=["0-99", "100-499", "500-999", "1k-4.9k", "5k+"]
        )
    )
    .groupby("impressions_bucket", observed=False)
    .agg(
        n=("review_priority", "size"),
        review_rate=("review_priority", "mean")
    )
    .reset_index()
)

signal_1

,impressions_bucket,n,review_rate
0,0-99,57536,0.468472
1,100-499,30550,0.527463
2,500-999,13244,0.419284
3,1k-4.9k,24580,0.331896
4,5k+,8328,0.270293


In [7]:
signal_2 = (
    baseline_df.assign(
        position_bucket=pd.cut(
            baseline_df["gsc_avg_position"],
            bins=[0, 3, 10, 20, 50, float("inf")],
            labels=["1-3", "3-10", "10-20", "20-50", "50+"],
            include_lowest=True
        )
    )
    .groupby("position_bucket", observed=False)
    .agg(
        n=("review_priority", "size"),
        review_rate=("review_priority", "mean")
    )
    .reset_index()
)

signal_2

,position_bucket,n,review_rate
0,1-3,17549,0.151063
1,3-10,67711,0.249147
2,10-20,26743,0.718394
3,20-50,17920,0.910156
4,50+,4315,0.923986


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Rule implementation

The baseline score is based only on February GSC average position.

Pages with worse average positions receive higher scores and are ranked earlier for review. The score is a transparent transformation of the observed position signal, with no fitted model or future-window information.

Each row also receives one reason code and one action label so that the ranked queue can be reviewed by a human.

In [8]:
queue = baseline_df.copy()

queue["score"] = queue["gsc_avg_position"]

queue["reason_code"] = np.where(
    queue["gsc_avg_position"] >= 10,
    "position_slipping",
    "lower_priority"
)

queue["action_label"] = np.where(
    queue["gsc_avg_position"] >= 10,
    "review",
    "monitor"
)

queue = queue.sort_values(
    ["score", "client_hash_id", "content_hash_id"],
    ascending=[False, True, True]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

print("Queue shape:", queue.shape)

queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action_label"
    ]
].head(10)

Queue shape: (134238, 11)


,rank,client_hash_id,content_hash_id,gsc_avg_position,score,reason_code,action_label
0,1,client_f623b01661d4bfe4,content_8d9e923e3a9805d6,285.000000,285.000000,position_slipping,review
1,2,client_62f4a7e64f5e0096,content_6d2c8230f397535e,242.973684,242.973684,position_slipping,review
2,3,client_3f0ce4d44fe94f3d,content_767c793957ee4c79,218.000000,218.000000,position_slipping,review
3,4,client_3ffa76342f366962,content_a8158dc3820ecc59,215.000000,215.000000,position_slipping,review
4,5,client_62f4a7e64f5e0096,content_79df0fb7300dcf24,212.394737,212.394737,position_slipping,review
5,6,client_62f4a7e64f5e0096,content_222d68683fa74d0f,203.195652,203.195652,position_slipping,review
6,7,client_62f4a7e64f5e0096,content_925b42e068cbdbc4,202.384615,202.384615,position_slipping,review
7,8,client_f623b01661d4bfe4,content_34ba23178a47632a,196.000000,196.000000,position_slipping,review
8,9,client_62f4a7e64f5e0096,content_90b3b342f423ac66,194.500000,194.500000,position_slipping,review
9,10,client_f623b01661d4bfe4,content_4e85428104dcbf9f,192.500000,192.500000,position_slipping,review


In [9]:
import os

output_path = "work/outputs/baseline_action_score.csv"

os.makedirs("work/outputs", exist_ok=True)

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action_label"
]

queue[output_columns].to_csv(output_path, index=False)

print(f"Saved ranked queue to: {output_path}")

Saved ranked queue to: work/outputs/baseline_action_score.csv


In [10]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


base_rate = queue["review_priority"].mean()

precision_10 = precision_at_k(
    queue["score"],
    queue["review_priority"],
    10
)

precision_50 = precision_at_k(
    queue["score"],
    queue["review_priority"],
    50
)

print(f"Base rate: {base_rate:.4f}")
print(f"Precision@10: {precision_10:.4f}")
print(f"Precision@50: {precision_50:.4f}")

Base rate: 0.4397
Precision@10: 0.7000
Precision@50: 0.6800


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
top20 = queue.head(20).copy()

top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action_label"
    ]
]

,rank,client_hash_id,content_hash_id,gsc_avg_position,score,reason_code,action_label
0,1,client_f623b01661d4bfe4,content_8d9e923e3a9805d6,285.000000,285.000000,position_slipping,review
1,2,client_62f4a7e64f5e0096,content_6d2c8230f397535e,242.973684,242.973684,position_slipping,review
2,3,client_3f0ce4d44fe94f3d,content_767c793957ee4c79,218.000000,218.000000,position_slipping,review
3,4,client_3ffa76342f366962,content_a8158dc3820ecc59,215.000000,215.000000,position_slipping,review
4,5,client_62f4a7e64f5e0096,content_79df0fb7300dcf24,212.394737,212.394737,position_slipping,review
5,6,client_62f4a7e64f5e0096,content_222d68683fa74d0f,203.195652,203.195652,position_slipping,review
6,7,client_62f4a7e64f5e0096,content_925b42e068cbdbc4,202.384615,202.384615,position_slipping,review
7,8,client_f623b01661d4bfe4,content_34ba23178a47632a,196.000000,196.000000,position_slipping,review
8,9,client_62f4a7e64f5e0096,content_90b3b342f423ac66,194.500000,194.500000,position_slipping,review
9,10,client_f623b01661d4bfe4,content_4e85428104dcbf9f,192.500000,192.500000,position_slipping,review


### Top-20 review

The queue prioritizes pages with the worst observed February GSC average position. Because the rule uses only this signal, all top-20 rows receive the same action and reason code. Confidence is based on the strength of the ranking signal, not on page-level business context that is unavailable in this anonymized slice.

| Rank | Action | Why it's here | Confidence | What would make it wrong |
|---:|---|---|---|---|
| 1 | Review | Highest observed February average position (285.0). | High | The position may be based on very few impressions, making the average unstable. |
| 2 | Review | Very poor February average position (243.0). | High | Low-volume observations could make the position unreliable. |
| 3 | Review | Very poor February average position (218.0). | High | Sparse GSC observations could make the average unstable. |
| 4 | Review | Very poor February average position (215.0). | High | The position may not represent enough search activity to justify review. |
| 5 | Review | Very poor February average position (212.4). | High | Low impression volume could make the average misleading. |
| 6 | Review | Very poor February average position (203.2). | High | The ranking may be driven by limited observations. |
| 7 | Review | Very poor February average position (202.4). | High | A small number of impressions could make the average position unstable. |
| 8 | Review | Very poor February average position (196.0). | High | The page may have insufficient search observations for a reliable average. |
| 9 | Review | Very poor February average position (194.5). | High | Sparse search activity could make this ranking misleading. |
| 10 | Review | Very poor February average position (192.5). | High | The position may be unstable if based on limited impressions. |
| 11 | Review | Very poor February average position (184.0). | High | Low-volume GSC data could weaken the signal. |
| 12 | Review | Very poor February average position (183.0). | High | A small observation base could make the average unreliable. |
| 13 | Review | Very poor February average position (181.0). | High | The average may be unstable with limited impressions. |
| 14 | Review | Very poor February average position (180.6). | High | Limited search observations could produce an unreliable average. |
| 15 | Review | Very poor February average position (175.5). | High | The position may be unstable if search volume is low. |
| 16 | Review | Very poor February average position (166.8). | High | Sparse observations could make this a weak review candidate. |
| 17 | Review | Very poor February average position (160.3). | High | Limited GSC observations could distort the average position. |
| 18 | Review | Very poor February average position (156.0). | High | Low observation volume could make the ranking less trustworthy. |
| 19 | Review | Very poor February average position (156.0). | High | The position may not be stable enough to justify manual review. |
| 20 | Review | Very poor February average position (155.0). | High | Limited impressions could make the average position misleading. |

In [12]:
top20_weakness = (
    queue.head(20)
    .merge(
        baseline_df[
            [
                "client_hash_id",
                "content_hash_id",
                "gsc_impressions",
                "gsc_clicks"
            ]
        ],
        on=["client_hash_id", "content_hash_id"],
        how="left"
    )
)

top20_weakness[
    [
        "rank",
        "gsc_avg_position",
        "gsc_impressions_x",
        "gsc_clicks_x",
        "action_label",
        "reason_code"
    ]
]

,rank,gsc_avg_position,gsc_impressions_x,gsc_clicks_x,action_label,reason_code
0,1,285.000000,1.0,0.0,review,position_slipping
1,2,242.973684,38.0,0.0,review,position_slipping
2,3,218.000000,1.0,1.0,review,position_slipping
3,4,215.000000,1.0,0.0,review,position_slipping
4,5,212.394737,38.0,0.0,review,position_slipping
5,6,203.195652,46.0,0.0,review,position_slipping
6,7,202.384615,39.0,0.0,review,position_slipping
7,8,196.000000,1.0,0.0,review,position_slipping
8,9,194.500000,42.0,0.0,review,position_slipping
9,10,192.500000,2.0,0.0,review,position_slipping


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

Several top-ranked rows are weak picks because their February average position is based on very few GSC impressions. For example, ranks 1, 3, 4, 8, 11, 12, 13, 15, 18, and 19 have only 1–2 impressions. A very high average position from such a small observation count may be unstable, so these rows should be treated as review candidates rather than confirmed problems.

### Leakage check

The baseline score uses only `gsc_avg_position` from the February feature window. It does not use March outcome fields, the `review_priority` label, or any product decision flags. March data is used only to construct and evaluate the outcome, not to rank the queue.

In [13]:
score_columns = [
    "score",
    "reason_code",
    "action_label",
    "rank"
]

leakage_columns = [
    "march_avg_position",
    "review_priority"
]

print("Score inputs:", ["gsc_avg_position"])
print("Leakage columns excluded from scoring:", leakage_columns)

assert queue["score"].equals(queue["gsc_avg_position"])
assert "march_avg_position" not in score_columns
assert "review_priority" not in score_columns

print("Leakage check passed.")

Score inputs: ['gsc_avg_position']
Leakage columns excluded from scoring: ['march_avg_position', 'review_priority']
Leakage check passed.


### Named limitation

**Low-impression instability:** The slice contains pages with very high average positions but only a handful of February impressions. This can make the position signal noisy, so the baseline should be treated as a review-prioritization tool rather than proof that a page has a problem.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.